#  Notebook 03 — Classification (Rain / No Rain)

Trains XGBoost classifier, evaluates, plots confusion matrix, ROC curve, feature importance.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

import warnings
from config_utils import load_config
from train_classifier import run_training

warnings.filterwarnings('ignore')
cfg = load_config(PROJECT_ROOT / 'configs' / 'config.yaml')
model, scaler, metrics = run_training(cfg['_meta']['config_path'])

## Results

In [ ]:
import json
with open(Path(cfg['paths']['reports']) / 'classifier_metrics.json') as f:
    m = json.load(f)
import pandas as pd
pd.DataFrame(m)

## Visualize Results

In [ ]:
import pandas as pd, numpy as np, joblib
from features import build_features_for_splits, get_feature_columns, apply_scaler, prepare_Xy
from visualize import plot_confusion_matrix, plot_roc_curve, plot_precision_recall, plot_feature_importance

train = pd.read_csv(cfg['paths']['train'], parse_dates=['date'])
val   = pd.read_csv(cfg['paths']['val'], parse_dates=['date'])
test  = pd.read_csv(cfg['paths']['test'], parse_dates=['date'])
_, _, test = build_features_for_splits(train=train, val=val, test=test)
feat_cols = get_feature_columns(cfg)
feat_cols = [c for c in feat_cols if c in test.columns]
clf = joblib.load(cfg['paths']['model_clf'])
sc  = joblib.load(cfg['paths']['scaler'])
X_test, y_test = prepare_Xy(test, feat_cols, 'rain_today')
X_test_sc = apply_scaler(X_test, sc)
y_pred  = clf.predict(X_test_sc)
y_proba = clf.predict_proba(X_test_sc)[:,1]
plot_confusion_matrix(y_test, y_pred, cfg)
plot_roc_curve(y_test, y_proba, cfg)
plot_precision_recall(y_test, y_proba, cfg)
plot_feature_importance(feat_cols, clf.feature_importances_, 'XGBoost Classifier', cfg)
print(f"Plots saved to {cfg['paths']['plots']}")